# HostWise - Day 2: Shared Modeling Foundation

This notebook creates the **single shared train/test split** that every Day 2 notebook uses. It is run **once**; all other notebooks reload the saved split with `hostwise_shared.load_shared_split()` and never create their own.

| It does | It does not |
|---|---|
| Load and validate `hostwise_5cities.csv` | Clean, impute, scale or encode anything |
| Apply the agreed modeling scope | Add outlier rules or other filters |
| Define the target and the eight initial features | Train or evaluate models |
| Create, verify and save one reproducible 80/20 split | Use the test set for any decision |
| Document the evaluation protocol | Decide how cities are weighted |

**Outputs** (in `../data/splits/`): `train_ids.csv`, `test_ids.csv`, `split_config.json`.

All agreed definitions (paths, scope, target, features, split settings, evaluation) live in **`hostwise_shared.py`**, which must sit in the same folder as this notebook. Keeping them in one module means the notebook that creates the split and the notebooks that reload it cannot drift apart.

## 1. Setup

In [ ]:
import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split

import hostwise_shared as hs

pd.set_option("display.float_format", "{:,.2f}".format)

In [ ]:
SPLIT_DIR = Path("../data/splits")
OVERWRITE_SPLIT = False   # keep False: an existing saved split is never replaced silently

In [ ]:
pd.Series({
    "dataset file": hs.DATA_FILENAME,
    "target": hs.TARGET,
    "numeric features": ", ".join(hs.NUMERIC_FEATURES),
    "categorical features": ", ".join(hs.CATEGORICAL_FEATURES),
    "test size": hs.TEST_SIZE,
    "random state": hs.RANDOM_STATE,
    "stratified by": "city",
}, name="agreed setting").to_frame()

## 2. Load and validate the dataset

`load_dataset()` reads the first existing path among `../data/processed/hostwise_5cities.csv`, `data/processed/...`, the current folder and `Airbnb/`. It stops with an error if the file does not contain exactly the five project cities or if any listing `id` is duplicated. Listing IDs are read as 64-bit integers so that these long IDs are never rounded.

The file's SHA-256 fingerprint is recorded in the configuration. Any later notebook that reloads the split checks this fingerprint and fails if a different version of the dataset is used.

In [ ]:
data_file = hs.find_data_file()
df = hs.load_dataset(data_file)

print(f"Loaded {data_file}: {df.shape[0]:,} rows x {df.shape[1]} columns")

In [ ]:
df["city"].value_counts().reindex(hs.CITY_ORDER).rename("listings").to_frame()

In [ ]:
dataset_sha256 = hs.file_sha256(data_file)
print(f"SHA-256: {dataset_sha256}")

## 3. Target: nightly price

The target `price_num` is the advertised nightly price parsed from the `price` text (e.g. `$1,234.00` -> 1234.0). The `$` sign is formatting only: prices are in the city's local currency (London GBP, other cities EUR).

A **valid price** means the value is present, parseable as a number and greater than 0. The "> 0" part is a sanity check, not a new filter: Day 1 found no prices <= 0. The table below confirms this before the scope is applied.

In [ ]:
price_num_all = hs.parse_price(df["price"])
is_active = df["source"] == hs.SCOPE_SOURCE

pd.Series({
    "active listings (city scrape)": is_active.sum(),
    "price missing": (is_active & df["price"].isna()).sum(),
    "price present but not parseable": (is_active & df["price"].notna() & price_num_all.isna()).sum(),
    "price <= 0": (is_active & (price_num_all <= 0)).sum(),
    "valid price": (is_active & (price_num_all > 0)).sum(),
}, name="listings").to_frame()

## 4. Modeling scope

The agreed scope is applied in three cumulative steps, defined once in `hs.scope_masks()`:

1. `source == "city scrape"` - listings present in the current scrape;
2. a valid price (section 3);
3. `minimum_nights <= 29` - short-stay listings. Listings with a missing `minimum_nights` cannot be shown to meet this condition and are excluded, exactly as in the Day 1 analysis.

**Not applied here** (each is still a pending team decision and, if adopted, belongs inside the training pipeline): price outlier fences or floors, removal of Hotel or Shared rooms, and removal of rows with missing feature values.

In [ ]:
cumulative_mask = pd.Series(True, index=df.index)
scope_steps = {"all listings": df["city"].value_counts()}

for step_name, step_mask in hs.scope_masks(df).items():
    cumulative_mask &= step_mask
    scope_steps[f"+ {step_name}"] = df.loc[cumulative_mask, "city"].value_counts()

scope_steps = pd.DataFrame(scope_steps).reindex(hs.CITY_ORDER).T
scope_steps["total"] = scope_steps.sum(axis=1)
scope_steps

In [ ]:
df_model = hs.apply_modeling_scope(df)

print(f"Modeling population: {len(df_model):,} listings (expected {hs.EXPECTED_SCOPE_ROWS:,})")
if len(df_model) != hs.EXPECTED_SCOPE_ROWS:
    raise ValueError(
        "The modeling population differs from the Day 1 count. Check that the correct "
        "hostwise_5cities.csv is used before creating the split."
    )

## 5. Target and initial features

The eight agreed initial features are kept **raw**. `city` is both a feature and the stratification variable. A guard checks that no target-derived or demand column (for example `price_quote_*`, `estimated_revenue_l365d`, `availability_*`) is among the features.

In [ ]:
leaking_features = set(hs.FEATURES) & set(hs.LEAKAGE_COLUMNS)
assert not leaking_features, f"Leakage columns in features: {leaking_features}"

model_data = df_model[[hs.ID_COLUMN] + hs.FEATURES + [hs.TARGET]]
model_data.dtypes.rename("dtype").to_frame()

In [ ]:
pd.DataFrame({
    "missing": model_data[hs.FEATURES].isna().sum(),
    "missing_%": model_data[hs.FEATURES].isna().mean() * 100,
})

Missing feature values are **kept**. Imputing them here, before the split, would let information from the test set leak into the training data. Track A handles them inside scikit-learn pipelines fitted on the training folds only.

## 6. Train/test split

**Agreed design:** a random 80/20 split of listings, stratified by `city`, with `random_state=42`. Stratification keeps each city's share the same in train and test, which matters because the cities are very unequal in size.

**Assumption made explicit:** rows are sorted by listing `id` before splitting. `train_test_split` depends on row order, so sorting makes the split independent of how the CSV happens to be ordered. The saved ID files remain the authoritative record of the split.

**Known limitation (documented, not changed):** the split is at listing level, so listings from the same host can appear in both train and test. Section 7 measures how often this happens.

In [ ]:
model_data = model_data.sort_values(hs.ID_COLUMN).reset_index(drop=True)

train_data, test_data = train_test_split(
    model_data,
    test_size=hs.TEST_SIZE,
    random_state=hs.RANDOM_STATE,
    stratify=model_data["city"],
)
print(f"train: {len(train_data):,} | test: {len(test_data):,}")

In [ ]:
X_train = train_data.set_index(hs.ID_COLUMN)[hs.FEATURES]
X_test = test_data.set_index(hs.ID_COLUMN)[hs.FEATURES]
y_train = train_data.set_index(hs.ID_COLUMN)[hs.TARGET]
y_test = test_data.set_index(hs.ID_COLUMN)[hs.TARGET]

print(f"X_train {X_train.shape} | X_test {X_test.shape} | y_train {y_train.shape} | y_test {y_test.shape}")

## 7. Verify the split

In [ ]:
train_ids = train_data[hs.ID_COLUMN]
test_ids = test_data[hs.ID_COLUMN]

assert train_ids.is_unique and test_ids.is_unique, "Duplicate IDs inside train or test"
assert set(train_ids).isdisjoint(test_ids), "Train and test share listing IDs"
assert set(train_ids) | set(test_ids) == set(df_model[hs.ID_COLUMN]), "Split does not cover the scope"
print("IDs unique, train/test disjoint, and together they cover the full modeling population.")

In [ ]:
city_shares = pd.DataFrame({
    "scope_%": model_data["city"].value_counts(normalize=True) * 100,
    "train_%": train_data["city"].value_counts(normalize=True) * 100,
    "test_%": test_data["city"].value_counts(normalize=True) * 100,
    "train_n": train_data["city"].value_counts(),
    "test_n": test_data["city"].value_counts(),
}).reindex(hs.CITY_ORDER)
city_shares

In [ ]:
host_of_listing = df_model.set_index(hs.ID_COLUMN)["host_id"]
train_hosts = set(host_of_listing.loc[train_ids])
test_hosts_seen_in_train = host_of_listing.loc[test_ids].isin(train_hosts)

print(f"Test listings whose host also has a listing in train: {test_hosts_seen_in_train.mean():.1%}")

**How to read the host check:** this share is informational. A high value means part of the test set comes from hosts the model has already seen, which can make test scores somewhat optimistic for multi-listing hosts. It does not change the agreed split; it is a caveat to report with the results.

## 8. Save the shared split

The split is saved as two ID files plus a configuration file. If a saved split already exists, it is **not overwritten**. Instead, the in-memory split is compared with the saved one. If they differ (for example because the data or a library version changed), the notebook stops so the team can investigate. The saved split stays authoritative. Set `OVERWRITE_SPLIT = True` only after a team decision to replace it.

In [ ]:
split_config = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "dataset": {
        "file_name": data_file.name,
        "path_used": str(data_file),
        "sha256": dataset_sha256,
        "shape": list(df.shape),
    },
    "scope": {
        "conditions": list(hs.scope_masks(df)),
        "rows": len(df_model),
        "not_applied": ["price outlier fences or floors", "room-type exclusions",
                        "removal of rows with missing feature values"],
    },
    "target": {
        "name": hs.TARGET,
        "definition": "advertised nightly price parsed from `price` (remove '$' and ','), local currency",
        "currency": hs.CURRENCY,
    },
    "features": {"numeric": hs.NUMERIC_FEATURES, "categorical": hs.CATEGORICAL_FEATURES},
    "split": {
        "method": "sklearn.model_selection.train_test_split on rows sorted by id",
        "test_size": hs.TEST_SIZE,
        "random_state": hs.RANDOM_STATE,
        "stratify": "city",
    },
    "n_train": len(train_ids),
    "n_test": len(test_ids),
    "city_counts": {"train": train_data["city"].value_counts().to_dict(),
                    "test": test_data["city"].value_counts().to_dict()},
    "evaluation": {
        "per_city": "MAE on the original price scale, local currency (GBP London, EUR others)",
        "cross_city_candidate": "MAE on log1p(price), reported both macro (equal city weight) and pooled",
        "primary_cross_city_aggregation": "pending team decision",
    },
    "library_versions": {"pandas": pd.__version__, "scikit-learn": sklearn.__version__},
}

In [ ]:
SPLIT_DIR.mkdir(parents=True, exist_ok=True)
split_exists = (SPLIT_DIR / hs.CONFIG_FILE).exists()

if split_exists and not OVERWRITE_SPLIT:
    saved_train_ids, saved_test_ids, _ = hs.read_split_ids(SPLIT_DIR)
    if set(saved_train_ids) != set(train_ids) or set(saved_test_ids) != set(test_ids):
        raise ValueError("A saved split exists and differs from this run. It was NOT overwritten - investigate first.")
    print(f"A saved split already exists in {SPLIT_DIR} and is identical to this run. Nothing overwritten.")
else:
    train_ids.to_frame().to_csv(SPLIT_DIR / hs.TRAIN_IDS_FILE, index=False)
    test_ids.to_frame().to_csv(SPLIT_DIR / hs.TEST_IDS_FILE, index=False)
    with open(SPLIT_DIR / hs.CONFIG_FILE, "w", encoding="utf-8") as handle:
        json.dump(split_config, handle, indent=2)
    print(f"Saved {hs.TRAIN_IDS_FILE}, {hs.TEST_IDS_FILE} and {hs.CONFIG_FILE} to {SPLIT_DIR}")

## 9. Reload check

This is the same call every teammate will use. `load_shared_split()` reloads the IDs and validates them: the dataset fingerprint and shape, the scope conditions, unique and disjoint IDs, and full coverage of the modeling population. It then rebuilds raw `X`/`y`. The result must be identical to the split created above.

In [ ]:
X_train_r, X_test_r, y_train_r, y_test_r = hs.load_shared_split(split_dir=SPLIT_DIR, data_path=data_file)

assert X_train_r.sort_index().equals(X_train.sort_index())
assert X_test_r.sort_index().equals(X_test.sort_index())
assert y_train_r.sort_index().equals(y_train.sort_index())
assert y_test_r.sort_index().equals(y_test.sort_index())
print("Reloaded split is identical to the saved split.")

## 10. Evaluation agreement

Every model, including baselines, is evaluated with the same function, `hs.evaluate_by_city(y_true, y_pred, city)`:

| Metric | Scale | Reported |
|---|---|---|
| `mae_local` | Original nightly price, local currency | Per city only - never averaged across cities, because GBP and EUR errors cannot be added |
| `mae_log` | `log1p(price)` - a relative error, comparable across currencies | Per city, plus two cross-city rows: **macro** (each city weighted equally) and **pooled** (each listing weighted equally) |

**Pending team decision:** which cross-city row is the primary model-selection metric. Macro gives Barcelona the same weight as London; pooled is dominated by London and Paris. Both are always reported until the team decides.

**Protocol for all models**
- Model selection and preprocessing choices use cross-validation on `X_train` / `y_train` only, with all preprocessing inside the pipeline so that it is refitted on each training fold.
- The test set is used **once**, for the final comparison of the selected models.
- Predictions are made on the price scale (convert back first if a model is trained on a log target); negative predictions are clipped to 0 before scoring.

## 11. How teammates use this foundation

**Requirements:** `hostwise_shared.py` in the same folder as your notebook, the team's `hostwise_5cities.csv` (same file version), and the saved files in `../data/splits/`.

**Track B (modeling): eight initial features**
```python
from hostwise_shared import load_shared_split, evaluate_by_city

X_train, X_test, y_train, y_test = load_shared_split()
# ... fit a pipeline with cross-validation on X_train / y_train only ...
results = evaluate_by_city(y_test, y_pred, X_test["city"])
```

**Track A (preprocessing): adding features without changing the split**
```python
from hostwise_shared import load_shared_frames

train_df, test_df = load_shared_frames()   # same listings, all original columns
# derive new columns (e.g. bath count from bathrooms_text) on these frames;
# fit any learned transformation on train_df only
```
Alternatively, `load_shared_split(features=[...])` returns any list of raw columns for the same rows. Leakage columns are refused.

**Rules**
- Never call `train_test_split` on the modeling data in another notebook.
- Never rerun this notebook with `OVERWRITE_SPLIT = True` without a team decision.
- If `load_shared_split()` raises a dataset-version error, you are using a different CSV - get the team's file rather than recreating the split.